# TP3 : Transfer Learning sur BloodMNIST (VGG16 & ResNet)

**Objectifs :**
- Charger et adapter une architecture pré-entraînée (VGG16) à une tâche médicale spécifique
- Appliquer deux modes de transfer learning : **Feature Extraction** et **Fine-Tuning**
- Maîtriser le dégel progressif des couches pour optimiser les performances sans overfitting
- Comparer VGG16 et ResNet sur la même tâche
- Manipuler des données médicales avec PyTorch (BloodMNIST, 8 classes, 224×224)

In [ ]:
# Installation (Google Colab)
!pip install medmnist -q

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import models

import medmnist
from medmnist import BloodMNIST

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"MedMNIST version : {medmnist.__version__}")

os.makedirs('models', exist_ok=True)

## 1. Chargement du dataset BloodMNIST (224×224, 8 classes)

BloodMNIST contient 17 092 images de microscopie de cellules sanguines réparties en 8 classes :
Basophil, Eosinophil, Erythroblast, Immature Granulocyte, Lymphocyte, Monocyte, Neutrophil, Platelet.

On utilise le format natif **224×224** pour rester compatible avec VGG16/ResNet pré-entraînés sur ImageNet.

In [ ]:
# Normalisation ImageNet (poids pré-entraînés)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# size=224 -> images natives 224x224 (compatibles VGG16 / ResNet)
train_dataset = BloodMNIST(split='train', transform=transform, download=True, size=224)
val_dataset   = BloodMNIST(split='val',   transform=transform, download=True, size=224)
test_dataset  = BloodMNIST(split='test',  transform=transform, download=True, size=224)

NUM_CLASSES = 8
BATCH_SIZE  = 16  # consigne du TP — testé aussi avec 8 et 32 plus bas

def make_loaders(batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(BATCH_SIZE)

print(f"Train : {len(train_dataset)} images")
print(f"Val   : {len(val_dataset)} images")
print(f"Test  : {len(test_dataset)} images")
print(f"Nombre de classes : {NUM_CLASSES}")

In [ ]:
# Visualisation : un exemple de chaque classe
class_names = ['Basophil', 'Eosinophil', 'Erythroblast', 'Imm. Granulocyte',
               'Lymphocyte', 'Monocyte', 'Neutrophil', 'Platelet']

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
labels_arr = train_dataset.labels.flatten()
for c in range(NUM_CLASSES):
    idx = np.where(labels_arr == c)[0][0]
    img, _ = train_dataset[idx]
    img_show = img.numpy().transpose(1, 2, 0)
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)  # dénormalisation
    img_show = np.clip(img_show, 0, 1)
    axes[c].imshow(img_show)
    axes[c].set_title(f"{c}: {class_names[c]}", fontsize=9)
    axes[c].axis('off')
plt.suptitle("Les 8 classes de BloodMNIST", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Fonctions utilitaires d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.squeeze().long().to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            total += labels.size(0)
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / total, acc, np.array(all_preds), np.array(all_labels)

def fit(model, train_loader, val_loader, epochs, lr, params_to_optimize=None, device=device, label=""):
    criterion = nn.CrossEntropyLoss()
    if params_to_optimize is None:
        params_to_optimize = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params_to_optimize, lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va, _, _ = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        print(f"[{label}] Epoch {epoch+1}/{epochs} | Train L:{tl:.4f} A:{ta:.4f} | Val L:{vl:.4f} A:{va:.4f}")
    return history

def test_model(model, test_loader, label=""):
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
    print(f"\n=== Test [{label}] === Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f}")
    print("\nRapport de classification :")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print("Matrice de confusion :")
    print(confusion_matrix(labels, preds))
    return test_acc

def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
    ax1.set_title(f'Loss {title}'); ax1.set_xlabel('Epoch'); ax1.legend()
    ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
    ax2.set_title(f'Accuracy {title}'); ax2.set_xlabel('Epoch'); ax2.legend()
    plt.tight_layout(); plt.show()

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 3. VGG16 — Feature Extraction

On charge VGG16 pré-entraîné sur ImageNet (1000 classes), on **gèle** tous les poids de l'extracteur de caractéristiques (`features` + `avgpool`), et on remplace la dernière couche du classifieur pour produire 8 sorties.

In [ ]:
def build_vgg16_feature_extractor(num_classes=NUM_CLASSES):
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    # Gel de tout l'extracteur de caractéristiques
    for p in model.features.parameters():
        p.requires_grad = False
    # Remplacement de la dernière couche FC (4096 -> num_classes)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)
    return model.to(device)

vgg_fe = build_vgg16_feature_extractor()
print(f"VGG16 (Feature Extraction) — Paramètres entraînables : {count_trainable(vgg_fe):,}")
print(f"VGG16 — Paramètres totaux                          : {sum(p.numel() for p in vgg_fe.parameters()):,}")

### 3.1 Entraînement initial (seule la tête de classification est apprise)

Convergence rapide attendue : on n'optimise que ~33M paramètres au lieu de 138M.

In [ ]:
EPOCHS_FE = 3   # convergence rapide
LR_FE     = 1e-3

history_vgg_fe = fit(vgg_fe, train_loader, val_loader,
                     epochs=EPOCHS_FE, lr=LR_FE, label="VGG16-FE")
plot_history(history_vgg_fe, "(VGG16 — Feature Extraction)")
acc_vgg_fe = test_model(vgg_fe, test_loader, label="VGG16-FE")

## 4. VGG16 — Fine-Tuning Niveau 1

On dégèle le **dernier bloc de convolution** (`features[24:]`, soit conv5_1, conv5_2, conv5_3 + ReLU + MaxPool) en plus de la tête, puis on ré-entraîne avec un **learning rate faible** (1e-4) pour ne pas dégrader les représentations pré-entraînées.

In [ ]:
# On repart du modèle FE déjà entraîné (la tête est déjà adaptée)
vgg_ft1 = copy.deepcopy(vgg_fe)

# Dégel du dernier bloc conv (indices 24 -> 30 dans features)
for i, layer in enumerate(vgg_ft1.features):
    layer.requires_grad_(i >= 24)

print(f"VGG16 (FT-1) — Paramètres entraînables : {count_trainable(vgg_ft1):,}")

EPOCHS_FT1 = 3
LR_FT1     = 1e-4   # LR faible pour le fine-tuning

history_vgg_ft1 = fit(vgg_ft1, train_loader, val_loader,
                      epochs=EPOCHS_FT1, lr=LR_FT1, label="VGG16-FT1")
plot_history(history_vgg_ft1, "(VGG16 — FT Niveau 1)")
acc_vgg_ft1 = test_model(vgg_ft1, test_loader, label="VGG16-FT1")

## 5. VGG16 — Fine-Tuning Niveau 2

On dégèle les **deux derniers blocs de convolution** (`features[17:]`, soit blocs 4 et 5). On augmente la capacité d'adaptation mais on s'expose davantage au surapprentissage.

In [ ]:
vgg_ft2 = copy.deepcopy(vgg_fe)

for i, layer in enumerate(vgg_ft2.features):
    layer.requires_grad_(i >= 17)

print(f"VGG16 (FT-2) — Paramètres entraînables : {count_trainable(vgg_ft2):,}")

EPOCHS_FT2 = 3
LR_FT2     = 1e-4

history_vgg_ft2 = fit(vgg_ft2, train_loader, val_loader,
                      epochs=EPOCHS_FT2, lr=LR_FT2, label="VGG16-FT2")
plot_history(history_vgg_ft2, "(VGG16 — FT Niveau 2)")
acc_vgg_ft2 = test_model(vgg_ft2, test_loader, label="VGG16-FT2")

## 6. Comparaison des stratégies VGG16

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for hist, name in [(history_vgg_fe,  'FE'),
                   (history_vgg_ft1, 'FT-1'),
                   (history_vgg_ft2, 'FT-2')]:
    ax1.plot(hist['val_loss'], label=name, marker='o')
    ax2.plot(hist['val_acc'],  label=name, marker='o')
ax1.set_title('Val Loss — VGG16'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Val Accuracy — VGG16'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nRécap VGG16 (Test Accuracy) :")
print(f"  Feature Extraction  : {acc_vgg_fe:.4f}")
print(f"  Fine-Tuning Niveau 1: {acc_vgg_ft1:.4f}")
print(f"  Fine-Tuning Niveau 2: {acc_vgg_ft2:.4f}")

**Analyse :**
- **Feature Extraction** converge le plus vite car peu de paramètres sont mis à jour ; les features ImageNet sont déjà très expressives sur des images de microscopie.
- **Fine-Tuning Niveau 1** apporte généralement un gain modéré : adapter le dernier bloc spécialise les features de plus haut niveau à la cytologie sans détruire les features bas-niveau.
- **Fine-Tuning Niveau 2** peut dépasser FT-1 si le dataset est suffisamment grand, mais on observe souvent un écart Train/Val plus marqué (overfitting), surtout avec peu d'époques et sans data augmentation.

## 7. Influence du batch size (BS=8 et BS=32) — Feature Extraction VGG16

In [ ]:
bs_results = {16: acc_vgg_fe}

for bs in [8, 32]:
    print(f"\n--- VGG16 Feature Extraction avec BS={bs} ---")
    tr_l, vl_l, ts_l = make_loaders(bs)
    m = build_vgg16_feature_extractor()
    fit(m, tr_l, vl_l, epochs=2, lr=LR_FE, label=f"VGG16-FE-BS{bs}")
    bs_results[bs] = test_model(m, ts_l, label=f"VGG16-FE-BS{bs}")

print("\nRécap selon le batch size (VGG16-FE) :")
for bs in sorted(bs_results):
    print(f"  BS={bs:>3} -> Test Accuracy = {bs_results[bs]:.4f}")

## 8. Refaire les expériences avec ResNet (ResNet18)

On utilise **ResNet18** : architecture résiduelle, beaucoup plus légère que VGG16 (~11M paramètres vs 138M) tout en restant très performante grâce aux skip connections.

- **Feature Extraction** : gel total sauf `fc` (Linear → 8 classes)
- **FT-1** : dégel de `layer4` (dernier bloc résiduel) + `fc`
- **FT-2** : dégel de `layer3` + `layer4` + `fc`

In [ ]:
def build_resnet18_feature_extractor(num_classes=NUM_CLASSES):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)  # requires_grad=True par défaut
    return model.to(device)

# Repartir des loaders BS=16
train_loader, val_loader, test_loader = make_loaders(16)

# 8.a Feature Extraction
resnet_fe = build_resnet18_feature_extractor()
print(f"ResNet18 (FE) — Paramètres entraînables : {count_trainable(resnet_fe):,}")
history_res_fe = fit(resnet_fe, train_loader, val_loader, epochs=3, lr=1e-3, label="ResNet-FE")
plot_history(history_res_fe, "(ResNet18 — FE)")
acc_res_fe = test_model(resnet_fe, test_loader, label="ResNet-FE")

In [ ]:
# 8.b Fine-Tuning Niveau 1 : dégel layer4
resnet_ft1 = copy.deepcopy(resnet_fe)
for p in resnet_ft1.layer4.parameters():
    p.requires_grad = True

print(f"ResNet18 (FT-1) — Paramètres entraînables : {count_trainable(resnet_ft1):,}")
history_res_ft1 = fit(resnet_ft1, train_loader, val_loader, epochs=3, lr=1e-4, label="ResNet-FT1")
plot_history(history_res_ft1, "(ResNet18 — FT Niveau 1)")
acc_res_ft1 = test_model(resnet_ft1, test_loader, label="ResNet-FT1")

In [ ]:
# 8.c Fine-Tuning Niveau 2 : dégel layer3 + layer4
resnet_ft2 = copy.deepcopy(resnet_fe)
for p in resnet_ft2.layer3.parameters():
    p.requires_grad = True
for p in resnet_ft2.layer4.parameters():
    p.requires_grad = True

print(f"ResNet18 (FT-2) — Paramètres entraînables : {count_trainable(resnet_ft2):,}")
history_res_ft2 = fit(resnet_ft2, train_loader, val_loader, epochs=3, lr=1e-4, label="ResNet-FT2")
plot_history(history_res_ft2, "(ResNet18 — FT Niveau 2)")
acc_res_ft2 = test_model(resnet_ft2, test_loader, label="ResNet-FT2")

## 9. Comparaison finale VGG16 vs ResNet18

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for hist, name in [(history_vgg_fe,  'VGG-FE'),
                   (history_vgg_ft1, 'VGG-FT1'),
                   (history_vgg_ft2, 'VGG-FT2'),
                   (history_res_fe,  'ResNet-FE'),
                   (history_res_ft1, 'ResNet-FT1'),
                   (history_res_ft2, 'ResNet-FT2')]:
    ax1.plot(hist['val_loss'], label=name, marker='o')
    ax2.plot(hist['val_acc'],  label=name, marker='o')
ax1.set_title('Val Loss — comparaison'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.set_title('Val Accuracy — comparaison'); ax2.set_xlabel('Epoch'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\n" + "="*60)
print(f"{'Stratégie':<30} {'Test Accuracy':>15}")
print("="*60)
print(f"{'VGG16  — Feature Extraction':<30} {acc_vgg_fe:>15.4f}")
print(f"{'VGG16  — Fine-Tuning Niv. 1':<30} {acc_vgg_ft1:>15.4f}")
print(f"{'VGG16  — Fine-Tuning Niv. 2':<30} {acc_vgg_ft2:>15.4f}")
print(f"{'ResNet18 — Feature Extraction':<30} {acc_res_fe:>15.4f}")
print(f"{'ResNet18 — Fine-Tuning Niv. 1':<30} {acc_res_ft1:>15.4f}")
print(f"{'ResNet18 — Fine-Tuning Niv. 2':<30} {acc_res_ft2:>15.4f}")
print("="*60)

**Discussion :**
- **ResNet18 vs VGG16 :** à nombre d'époques équivalent, ResNet18 atteint souvent une accuracy comparable ou supérieure tout en étant ~12× plus léger en paramètres. Les *skip connections* facilitent l'optimisation et limitent le risque de gradient qui s'évanouit.
- **Niveau de fine-tuning :** dégeler progressivement révèle un compromis classique entre *capacité d'adaptation* (plus de couches dégelées = plus d'expressivité) et *risque de surapprentissage* (plus de paramètres à apprendre sur un dataset limité).
- **Batch size :** BS plus petit → mises à jour plus bruitées (effet régularisant léger) mais entraînement plus lent par époque ; BS plus grand → estimations de gradient plus stables mais consommation mémoire plus élevée.

## 10. Enregistrement des modèles

In [ ]:
torch.save(vgg_fe.state_dict(),   'models/vgg16_fe.pth')
torch.save(vgg_ft1.state_dict(),  'models/vgg16_ft1.pth')
torch.save(vgg_ft2.state_dict(),  'models/vgg16_ft2.pth')
torch.save(resnet_fe.state_dict(),  'models/resnet18_fe.pth')
torch.save(resnet_ft1.state_dict(), 'models/resnet18_ft1.pth')
torch.save(resnet_ft2.state_dict(), 'models/resnet18_ft2.pth')

print("Modèles enregistrés dans le dossier 'models/' :")
for f in sorted(os.listdir('models')):
    size_mb = os.path.getsize(os.path.join('models', f)) / (1024*1024)
    print(f"  {f:25s}  ({size_mb:6.2f} MB)")